# Redox Agent — Evaluation Guide

This notebook walks through how to evaluate the **agent-redox-openai** Databricks App, which uses the Redox MCP server to interact with the Redox Engine platform API.

## What the evaluations test

The evaluation suite uses MLflow's `ConversationSimulator` to simulate multi-turn conversations with the agent across **8 operational scenarios** that mirror real Redox platform workflows:

| # | Scenario | Persona | What it validates |
|---|----------|---------|-------------------|
| 1 | List environments | Integration engineer | Agent can discover available Redox environments |
| 2 | Set active environment | Integration engineer | Agent can select and confirm an environment |
| 3 | List destinations | Technical PM | Agent can enumerate configured integration endpoints |
| 4 | Create a destination | Developer | Agent can create a new destination with required params |
| 5 | List test files | QA engineer | Agent can discover available sample payloads |
| 6 | Send test files | QA engineer | Agent can transmit test data to a destination |
| 7 | Access transaction logs | Integration engineer | Agent can retrieve and filter transaction history |
| 8 | Summarize transaction logs | Project manager | Agent can produce human-readable log summaries |

## Scorers

Each conversation is evaluated by 9 MLflow LLM judges:

- **Completeness** — Did the agent fully answer the question?
- **ConversationCompleteness** — Was the overall goal achieved across turns?
- **ConversationalSafety** — Was the conversation free of harmful content?
- **KnowledgeRetention** — Did the agent maintain context across turns?
- **UserFrustration** — Did the simulated user show signs of frustration?
- **Fluency** — Was the agent's language natural and well-structured?
- **RelevanceToQuery** — Were responses on-topic?
- **Safety** — General safety check on individual responses
- **ToolCallCorrectness** — Did the agent call the right MCP tools?

## Prerequisites

Before running evaluations, ensure:

1. **The agent app is deployed and running** — `agent-redox-openai` Databricks App is active
2. **MCP permissions granted** — The agent app's service principal has `CAN_USE` on `mcp-redox`
3. **MLflow experiment exists** — The experiment at `/Workspace/.experiments/agent-redox-openai-{target}` is created (done automatically by `databricks bundle deploy`)
4. **Network access** — This notebook's compute can reach the agent's endpoint and the Redox MCP server

In [0]:
%pip install databricks-openai openai-agents mlflow>=3.10.0 databricks-agents>=1.9.3 nest_asyncio python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
import mlflow

# Point MLflow to the same experiment the deployed agent uses.
# Update the target suffix (dev, prod, himss2026) to match your deployment.
EXPERIMENT_PATH = "/Workspace/.experiments/agent-redox-openai-himss2026"

mlflow.set_experiment(EXPERIMENT_PATH)
print(f"MLflow experiment: {EXPERIMENT_PATH}")

## Option A — Run via CLI (recommended for CI/CD)

The simplest way to run the full evaluation suite is from a terminal in the project directory:

```bash
cd /Workspace/Users/matthew.giglia@databricks.com/synthea-on-fhir/redox_agent/agent-redox-openai-sdk
uv run agent-evaluate
```

This invokes `agent_server/evaluate_agent.py:evaluate()` which:
1. Imports the agent (registers the `@invoke` function)
2. Creates a `ConversationSimulator` with 8 test cases
3. Runs `mlflow.genai.evaluate()` with all 9 scorers
4. Logs results to the configured MLflow experiment

## Option B — Run from this notebook

You can also run the evaluation directly from this notebook. The cells below reproduce the evaluation logic from `evaluate_agent.py` so you can inspect and modify individual test cases interactively.

In [0]:
import sys
import os

# Add the agent project to the Python path so we can import agent_server
PROJECT_DIR = "/Workspace/Users/matthew.giglia@databricks.com/synthea-on-fhir/redox_agent/agent-redox-openai-sdk"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [0]:
import asyncio
import logging
import nest_asyncio

from mlflow.genai.agent_server import get_invoke_function
from mlflow.genai.scorers import (
    Completeness,
    ConversationalSafety,
    ConversationCompleteness,
    Fluency,
    KnowledgeRetention,
    RelevanceToQuery,
    Safety,
    ToolCallCorrectness,
    UserFrustration,
)
from mlflow.genai.simulators import ConversationSimulator
from mlflow.types.responses import ResponsesAgentRequest

logging.getLogger("mlflow.utils.autologging_utils").setLevel(logging.ERROR)

# Import the agent to register the @invoke function
import agent_server.agent  # noqa: F401

print("Agent imported and @invoke function registered.")

In [0]:
# ---------------------------------------------------------------------------
# Redox Engine Platform API – Operational Evaluation Scenarios
# Each test case targets a specific Redox platform workflow that the agent
# should be able to accomplish via its MCP tools.
# ---------------------------------------------------------------------------
test_cases = [
    # 1. List environments
    {
        "goal": "Discover what Redox environments are available in the organization",
        "persona": "An integration engineer onboarding onto a new Redox project. You need to see what environments exist before doing any configuration.",
        "simulation_guidelines": [
            "Ask the agent to list all available Redox environments.",
            "Ask what the difference is between the environments returned (e.g., development vs staging vs production).",
            "Prefer short, direct questions.",
        ],
    },
    # 2. Set the active environment
    {
        "goal": "Select a specific Redox environment and confirm it is active for subsequent operations",
        "persona": "An integration engineer who has just reviewed the environment list and wants to work in a development or sandbox environment.",
        "simulation_guidelines": [
            "Ask the agent to set the active environment to a development or sandbox environment.",
            "Confirm which environment is now active.",
            "Ask whether subsequent tool calls will use the selected environment.",
        ],
    },
    # 3. List destinations
    {
        "goal": "List the destinations configured in the current Redox environment",
        "persona": "A technical project manager reviewing existing integration endpoints before adding a new one.",
        "simulation_guidelines": [
            "Ask the agent to list all destinations in the current environment.",
            "Ask for details about one of the destinations returned (e.g., its name, ID, status).",
            "Ask what types of destinations Redox supports.",
        ],
    },
    # 4. Create a new destination
    {
        "goal": "Create a new destination in Redox for receiving clinical data",
        "persona": "A developer setting up a new integration endpoint for a partner health system.",
        "simulation_guidelines": [
            "Ask the agent to create a new destination with a descriptive name like 'Test Clinic Inbound'.",
            "Ask what parameters are required to create a destination.",
            "Confirm the destination was created and ask for its ID.",
        ],
    },
    # 5. List available test files
    {
        "goal": "Discover what test files or sample payloads are available to send to a destination",
        "persona": "A QA engineer preparing to test a newly created destination by sending sample data.",
        "simulation_guidelines": [
            "Ask the agent to list available test files that can be sent to a destination.",
            "Ask what data types or event types the test files cover (e.g., ADT, ORU, scheduling).",
            "Ask which test file would be appropriate for testing a patient admission workflow.",
        ],
    },
    # 6. Send test files to a destination
    {
        "goal": "Send one or more test files to a destination and confirm they were transmitted",
        "persona": "A QA engineer ready to execute an end-to-end test by sending sample data through Redox.",
        "simulation_guidelines": [
            "Ask the agent to send a test file to the destination.",
            "Ask for confirmation that the test file was sent successfully.",
            "Ask if there is a way to verify the destination received the data.",
        ],
    },
    # 7. Access transaction logs
    {
        "goal": "Access and retrieve transaction logs for recent activity, such as the test file send",
        "persona": "An integration engineer troubleshooting a data exchange and needing to review what was sent and received.",
        "simulation_guidelines": [
            "Ask the agent to pull up the transaction logs for recent activity.",
            "Ask to filter or find logs related to the test file that was just sent.",
            "Ask about the status of the transactions (success, failure, pending).",
        ],
    },
    # 8. Review and summarize transaction logs
    {
        "goal": "Get a human-readable summary of transaction log entries to understand what happened",
        "persona": "A project manager who needs a concise summary of recent transaction activity for a status report.",
        "simulation_guidelines": [
            "Ask the agent to summarize the recent transaction logs.",
            "Ask for a breakdown of how many transactions succeeded vs failed.",
            "Ask the agent to highlight any errors or issues that need attention.",
            "Prefer a summary format suitable for sharing with non-technical stakeholders.",
        ],
    },
]

print(f"Defined {len(test_cases)} evaluation scenarios.")
for i, tc in enumerate(test_cases, 1):
    print(f"  {i}. {tc['goal'][:80]}")

In [0]:
import requests
from databricks.sdk import WorkspaceClient

# Call the deployed agent's HTTP endpoint instead of running locally.
# Running locally fails because serverless notebook compute cannot
# authenticate to the Redox MCP Databricks App.
AGENT_APP_URL = "https://agent-redox-openai-7474657999482942.aws.databricksapps.com"

w = WorkspaceClient()
token = w.config.token

def predict_fn(input: list[dict], **kwargs) -> dict:
    response = requests.post(
        f"{AGENT_APP_URL}/invocations",
        json={"input": input},
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
        timeout=300,
    )
    response.raise_for_status()
    return response.json()

print(f"predict_fn configured to call deployed agent at {AGENT_APP_URL}")

In [0]:
# Run the full evaluation suite
# This will simulate 8 multi-turn conversations (up to 6 turns each)
# and score them with 9 LLM judges.
#
# Expected runtime: ~10-20 minutes depending on model latency.

import mlflow

results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=predict_fn,
    scorers=[
        Completeness(),
        ConversationCompleteness(),
        ConversationalSafety(),
        KnowledgeRetention(),
        UserFrustration(),
        Fluency(),
        RelevanceToQuery(),
        Safety(),
        ToolCallCorrectness(),
    ],
)

print("Evaluation complete. View results in the MLflow experiment UI.")

In [0]:
# Display the evaluation metrics summary
display(results.metrics)

# Display per-row results (one row per simulated conversation)
display(results.tables["eval_results"])

## Customizing Evaluations

### Adding a new test case

Append a new dictionary to `test_cases` above with:

- **`goal`** — What the simulated user is trying to accomplish
- **`persona`** — Who they are (affects question style and expertise level)
- **`simulation_guidelines`** — Step-by-step instructions for the simulator's conversation flow

### Running a subset of scenarios

To run only specific test cases, slice the list before creating the simulator:

```python
# Run only the first 3 scenarios
simulator = ConversationSimulator(
    test_cases=test_cases[:3],
    max_turns=6,
    user_model="databricks:/databricks-claude-sonnet-4-5",
)
```

### Adjusting conversation depth

Change `max_turns` to control how many back-and-forth exchanges each scenario simulates. Higher values test knowledge retention but increase runtime.

### Adding custom scorers

See the [MLflow custom scorers documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers) to add domain-specific judges, e.g., a scorer that checks whether the agent correctly references Redox-specific terminology.